In [1]:
import json
from bs4 import BeautifulSoup
import pandas as pd
import os

# Parse Orthosphere Data

In [2]:
with open('orthosphere_data_raw.json', 'r') as file:
    orthosphere_data = json.load(file)

## Explore Schema

In [3]:
orthosphere_data.keys()

dict_keys(['2012/2', '2012/3', '2012/4', '2012/5', '2012/6', '2012/7', '2012/8', '2012/9', '2012/10', '2012/11', '2012/12', '2013/1', '2013/2', '2013/3', '2013/4', '2013/5', '2013/6', '2013/7', '2013/8', '2013/9', '2013/10', '2013/11', '2013/12', '2014/1', '2014/2', '2014/3', '2014/4', '2014/5', '2014/6', '2014/7', '2014/8', '2014/9', '2014/10', '2014/11', '2014/12', '2015/1', '2015/2', '2015/3', '2015/4', '2015/5', '2015/6', '2015/7', '2015/8', '2015/9', '2015/10', '2015/11', '2015/12', '2016/1', '2016/2', '2016/3', '2016/4', '2016/5', '2016/6', '2016/7', '2016/8', '2016/9', '2016/10', '2016/11', '2016/12', '2017/1', '2017/2', '2017/3', '2017/4', '2017/5', '2017/6', '2017/7', '2017/8', '2017/9', '2017/10', '2017/11', '2017/12', '2018/1', '2018/2', '2018/3', '2018/4', '2018/5', '2018/6', '2018/7', '2018/8', '2018/9', '2018/10', '2018/11', '2018/12', '2019/1', '2019/2', '2019/3', '2019/4', '2019/5', '2019/6', '2019/7', '2019/8', '2019/9', '2019/10', '2019/11', '2019/12', '2020/1', '2020

In [4]:
orthosphere_data['2012/2'][0].keys()

dict_keys(['date', 'url', 'article', 'comments'])

In [5]:
orthosphere_data['2012/2'][0]['article']['date']

'2012-02-29T17:43:25+00:00'

In [6]:
orthosphere_data['2012/2'][0]['article'].keys()

dict_keys(['title', 'date', 'author', 'html', 'text'])

In [7]:
orthosphere_data['2012/2'][0]['article']['author']

'Proph'

In [8]:
html = orthosphere_data['2012/2'][0]['article']['html']

In [9]:
t = BeautifulSoup(html)

In [10]:
[p.text for p in t.findAll('p')]

['Two Australian ethicists have declared their support for literal, unquestionable infanticide (h/t to Thinking Housewife):',
 'Two ethicists working with Australian universities argue in the latest online edition of the Journal of Medical Ethics that if abortion of a fetus is allowable, so to should be the termination of a newborn.',
 'Alberto Giubilini\xa0with Monash University in Melbourne\xa0and\xa0Francesca Minerva at the Centre for Applied Philosophy and Public Ethics at the University of Melbourne\xa0write that in “circumstances occur[ing] after birth such that they would have justified abortion, what we call after-birth abortion should be permissible.”',
 'The two are quick to note that they prefer the term “after-birth abortion“ as opposed to ”infanticide.” Why? Because it “[emphasizes] that the moral status of the individual killed is comparable with that of a fetus (on which ‘abortions’ in the traditional sense are performed) rather than to that of a child.” The authors also

In [11]:
orthosphere_data['2012/2'][0]['comments']

[{'author': 'Alice Teller',
  'text': 'I have a little scenario I propose to those who take the my body, my choice position. Imagine you are Under these circumstances would you force the remaining woman to nurse a child that is not her own, even if she didn’t want to? She is the only person there that can save the baby’s life. It makes for some interesting silences.\n',
  'time': '2012-02-29T18:07:21+00:00'},
 {'author': 'Sage',
  'text': 'I think you left a sentence out.  I’d be interested to read the rest of what you had to say.\n',
  'time': '2012-03-01T00:12:32+00:00'},
 {'author': 'Alice Teller',
  'text': 'Sorry, Imagine that  there is in a plane crash days away from help. Two nursing mothers, two babies. One mother dies, leaving an infant behind. One baby dies, leaving a grieving mother who refuses to nurse an infant that is not her own. She is the only hope to save the orphan infant. Would you force her to nurture the infant who will otherwise die without her. Does my body, my 

## Parse

### Define Model Schema

In [12]:
corpus_df = pd.DataFrame({
    'corpusID': [0],
    'corpusName': ['orthosphere'],
    
})

In [13]:
author_df = pd.DataFrame(columns=[
    'authorID',
    'corpusID',
    'name'
])

In [14]:
text_df = pd.DataFrame(columns = [
    'textID',
    'corpusID',
    'authorID',
    'title',
    'date',
    
])

In [15]:
paragraph_df = pd.DataFrame(columns = [
    'textID',
    'paragraphID',
    'order',
    'content'
])

In [16]:
comment_df = pd.DataFrame(columns = [
    'textID',
    'commentID',
    'authorID',
    'order',
    'content',
    'date'
])

### Parse

In [17]:
corp_id = 0

In [18]:
# extract info
auth_records = []
author_dict = {}
text_records = []
para_records = []
comment_records = []

auth_id = 0
rec_id = 0

for month in orthosphere_data.keys():
    print(month)
    for record in orthosphere_data[month]:
        
        if 'article' not in record.keys():
            continue
        
        author = record['article']['author']
        
        if author not in author_dict.keys():
            auth_records.append({
                'name': author,
                'authorID': auth_id,
                'corpusID': corp_id
            })
            
            author_dict[author] = auth_id
            
            text_auth_id = auth_id
            
            auth_id += 1
            
        else:
            text_auth_id = author_dict[author]
            
            
        
        text_records.append({
            'textID': rec_id,
            'authorID': text_auth_id,
            'title': record['article']['title'],
            'date': record['article']['date'],
            'corpusID': corp_id
        })
        
        # paragraph contents
        
        bs = BeautifulSoup(record['article']['html'])
        para_i = 0
        for para in bs.findAll('p'):
            para_records.append({
                'textID': rec_id,
                'paragraphID': f"{rec_id}_{para_i}",
                'order': para_i,
                'content': para.text
            })
            
            para_i += 1
            
        # comments
        ci = 0
        for c in record['comments']:
            c_auth = c['author']
            if c_auth not in author_dict.keys():
                auth_records.append({
                    'name': c_auth,
                    'authorID': auth_id,
                    'corpusID': corp_id
                })
                
                author_dict[c_auth] = auth_id
                
                c_auth_id = auth_id
                
                auth_id += 1
                
            else:
                c_auth_id = author_dict[c_auth]
                
            comment_records.append({
                'textID': rec_id,
                'commentID': f"{rec_id}_{ci}",
                'order': ci,
                'authorID': c_auth_id,
                'content': c['text'],
                'date': c['time']
            })
            ci += 1
            
        
        rec_id += 1
        
        
        
        

2012/2
2012/3
2012/4
2012/5
2012/6
2012/7
2012/8
2012/9
2012/10
2012/11
2012/12
2013/1
2013/2
2013/3
2013/4
2013/5
2013/6
2013/7
2013/8
2013/9
2013/10
2013/11
2013/12
2014/1
2014/2
2014/3
2014/4
2014/5
2014/6
2014/7
2014/8
2014/9
2014/10
2014/11
2014/12
2015/1
2015/2
2015/3
2015/4
2015/5
2015/6
2015/7
2015/8
2015/9
2015/10
2015/11
2015/12
2016/1
2016/2
2016/3
2016/4
2016/5
2016/6
2016/7
2016/8
2016/9
2016/10
2016/11
2016/12
2017/1
2017/2
2017/3
2017/4
2017/5
2017/6
2017/7
2017/8
2017/9
2017/10
2017/11
2017/12
2018/1
2018/2
2018/3
2018/4
2018/5
2018/6
2018/7
2018/8
2018/9
2018/10
2018/11
2018/12
2019/1
2019/2
2019/3
2019/4
2019/5
2019/6
2019/7
2019/8
2019/9
2019/10
2019/11
2019/12
2020/1
2020/2
2020/3
2020/4
2020/5
2020/6
2020/7
2020/8
2020/9
2020/10
2020/11
2020/12
2021/1
2021/2
2021/3
2021/4
2021/5
2021/6
2021/7
2021/8
2021/9
2021/10
2021/11
2021/12
2022/1
2022/2
2022/3
2022/4
2022/5
2022/6
2022/7
2022/8
2022/9
2022/10
2022/11
2022/12
2023/1
2023/2
2023/3
2023/4
2023/5
2023/6
2023/7
2

In [19]:
author_df = pd.concat(
    [author_df,
    pd.DataFrame(auth_records)]
)

text_df = pd.concat(
    [text_df,
    pd.DataFrame(text_records)]
)

paragraph_df = pd.concat(
    [paragraph_df,
    pd.DataFrame(para_records)]
)

comment_df = pd.concat([
    comment_df,
    pd.DataFrame(comment_records)
])

In [20]:
out_dir = 'model/'
os.makedirs(out_dir, exist_ok=True)

In [21]:
corpus_df.to_excel(out_dir + 'corpuses.xlsx', index = False)
author_df.to_excel(out_dir + 'authors.xlsx', index=False)
text_df.to_excel(out_dir + 'texts.xlsx', index = False)

paragraph_df.to_parquet(out_dir + 'paragraphs.parquet', index=False)
comment_df.to_parquet(out_dir + 'comments.parquet', index=False)
